In [4]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
import pickle

In [5]:
input_data = pd.read_csv("Churn_Modelling.csv")
input_data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [6]:
## Drop irrelevant columns
input_data = input_data.drop(['RowNumber','CustomerId','Surname'],axis=1)
input_data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [7]:
## Label encode gender
label_encode_gender = LabelEncoder()
input_data['Gender'] = label_encode_gender.fit_transform(input_data['Gender'])
input_data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [8]:
input_data[['Geography']] ## 2D array

,Geography
0,France
1,Spain
2,France
3,France
4,Spain
...,...
9995,France
9996,France
9997,France
9998,Germany


In [9]:
## one hot encode geography
onehot_encode_geo = OneHotEncoder()
onehot_encoded = onehot_encode_geo.fit_transform(input_data[['Geography']])
onehot_encoded.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [10]:
onehot_encode_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [11]:
geo_df = pd.DataFrame(onehot_encoded.toarray(),columns=onehot_encode_geo.get_feature_names_out(['Geography']))
geo_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [12]:
input_data = pd.concat([input_data.drop('Geography',axis=1),geo_df],axis=1)
input_data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [13]:
X = input_data.drop('EstimatedSalary',axis=1)
y = input_data['EstimatedSalary']

## split the dataset into training & testing

X_train, X_test, y_train , y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## Scale these features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [14]:
## Save encoders and scalers as serialized pickle file in binary format
## wb = write binary mode
with open("reg_label_encoder_gender.pkl",'wb') as file:
    pickle.dump(label_encode_gender,file)

with open("reg_onehot_encoder_geo.pkl", 'wb') as file:
    pickle.dump(onehot_encode_geo,file)

with open("reg_scaler.pkl",'wb') as file:
    pickle.dump(scaler,file)

## ANN Regression problem statement

In [16]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [17]:
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1), # default activation function is linear
])
# for regression problem the activation function for output should be linear

In [19]:
## compile the model
model.compile(optimizer='adam',loss='mean_absolute_error', metrics=['mae'])
# loss function for regression will be mean_absolute_error
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [20]:
## Set up the TensorBoard
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime
log_dir = 'regression_logs/fit/' + datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
tensorflow_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)

In [21]:
## Set up the Early Stopping
early_stopping_callback = EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [22]:
## Train the model
history = model.fit(
    X_train, y_train, validation_data=(X_test,y_test), epochs=100,
    callbacks = [tensorflow_callback,early_stopping_callback]
)

Epoch 1/100


250/250 [==============================] - 7s 12ms/step - loss: 100375.2656 - mae: 100375.2656 - val_loss: 98508.5156 - val_mae: 98508.5156
Epoch 2/100
250/250 [==============================] - 1s 6ms/step - loss: 99577.8047 - mae: 99577.8047 - val_loss: 96891.0859 - val_mae: 96891.0859
Epoch 3/100
250/250 [==============================] - 2s 9ms/step - loss: 96750.7266 - mae: 96750.7266 - val_loss: 92751.7734 - val_mae: 92751.7734
Epoch 4/100
250/250 [==============================] - 2s 10ms/step - loss: 91221.6250 - mae: 91221.6250 - val_loss: 85907.2344 - val_mae: 85907.2344
Epoch 5/100
250/250 [==============================] - 1s 3ms/step - loss: 83242.0078 - mae: 83242.0078 - val_loss: 77153.7500 - val_mae: 77153.7500
Epoch 6/100
250/250 [==============================] - 2s 7ms/step - loss: 73997.2031 - mae: 73997.2031 - val_loss: 68229.4141 - val_mae: 68229.4141
Epoch 7/100
250/250 [==============================] - 1s 4ms/step - loss: 65171.5273 - mae: 65171.5

In [23]:
## Load tensorboard extension
%load_ext tensorboard

In [25]:
%tensorboard --logdir regression_logs/fit

Reusing TensorBoard on port 6006 (pid 10968), started 0:01:19 ago. (Use '!kill 10968' to kill it.)

In [26]:
## Evaluate model on the test data
test_loss, test_mae = model.evaluate(X_test,y_test)
print(f"Test loss = {test_loss}")
print(f"Test MAE = {test_mae}")

63/63 [==============================] - 0s 2ms/step - loss: 50256.8945 - mae: 50256.8945
Test loss = 50256.89453125
Test MAE = 50256.89453125


In [27]:
model.save('regression_model.h5')

d:\Study_Content\LLM\ANN Classification\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
